# Roof Area Clustering per Building Type
**LOD2 3D Roof Dataset | South + East-West + Flat | No Fraction Applied**

Clusters real roof areas from the 3D LOD2 scan of NRW buildings.
No 20% fraction — total areas used directly. SESMG decides how much to use.

| Step | What it does |
|------|--------------|
| 1 | Imports |
| 2 | Configuration |
| 3 | Load M1 parquet |
| 4 | Load LOD2 roof dataset |
| 5 | Join by building ID |
| 6 | Roof classification |
| 7 | Distribution overview |
| 8 | Helper functions |
| 9 | Cluster per building type |
| 10 | Combination summary |
| 11 | Gap analysis |
| 12 | Save Excel |

---
## Step 1 — Imports

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


---
## Step 2 — Configuration

- `FLAT_TILT_THRESHOLD` — buildings below 10 degrees tilt = flat roof
  (consistent with Testsolar government platform definition)
- `K_MIN / K_MAX` — range of k tested by elbow
- `SAMPLE_N` — 30,000 buildings sampled for k-selection speed
- `RANDOM_STATE` — fixed seed for reproducibility

In [ ]:
BASE_DIR   = os.getcwd()
FILE_M1    = os.path.join(BASE_DIR, 'DEA_method1_final.parquet') # only ID and size_class used
FILE_ROOF  = os.path.join(BASE_DIR, 'lod2_nrw_roofs.geoparquet')
OUTPUT_DIR = os.path.join(BASE_DIR, 'clustering_results')

FLAT_TILT_THRESHOLD = 10   # degrees — below this = flat roof
K_MIN        = 2
K_MAX        = 10
SAMPLE_N     = 30_000
RANDOM_STATE = 42
SIZE_CLASSES = ['SFH', 'TH', 'MFH', 'AB']

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'BASE_DIR       : {BASE_DIR}')
print(f'FILE_M1 exists : {os.path.exists(FILE_M1)}')
print(f'FILE_ROOF exists: {os.path.exists(FILE_ROOF)}')
print('Configuration ready')

BASE_DIR       : c:\Users\zaito\Downloads\clustering
FILE_M1 exists : True
FILE_ROOF exists: True
Configuration ready


---
## Step 3 — Load M1 Parquet

Only building ID and size class are needed from the parquet.
The roof clustering is independent of the electricity method
so results apply equally to both M1 and M3b.

In [5]:
BASE_DIR   = os.getcwd()  # folder where this notebook is located
m1 = pd.read_parquet(FILE_M1, columns=[
    'id', 'size_class', 'area_m2', 'refurbishment_state'
])
print(f'M1 parquet loaded : {len(m1):,} buildings')
print(f'\nCount per building type:')
print(m1['size_class'].value_counts().to_string())

M1 parquet loaded : 4,133,323 buildings

Count per building type:
size_class
SFH    2723098
MFH     897247
TH      507633
AB        5345


---
## Step 4 — Load LOD2 Roof Dataset

The LOD2 (Level of Detail 2) dataset contains 3D roof geometry
extracted from the NRW building scan.

Columns used:
- `avg_tilt_deg` — average tilt angle per building (flat roof detection)
- `area_south_m2` — south facing roof area
- `area_east_m2` / `area_west_m2` — east and west facing areas
- `area_total_m2` — total roof area (used for flat roofs)

North facing area is excluded throughout.

In [6]:
BASE_DIR   = os.getcwd()  # folder where this notebook is located
roofs = gpd.read_parquet(FILE_ROOF)
roofs = roofs.drop(columns=[
    c for c in ['tile', 'roof_type_code', 'function',
                'gemeinde', 'creation_date', 'geometry']
    if c in roofs.columns
])
print(f'Roof dataset loaded: {len(roofs):,} buildings')
print(f'Columns: {list(roofs.columns)}')
roofs.head(3)

Roof dataset loaded: 5,097,348 buildings
Columns: ['building_id', 'roof_type_osm', 'height_m', 'avg_tilt_deg', 'area_north_m2', 'area_east_m2', 'area_south_m2', 'area_west_m2', 'area_total_m2']


,building_id,roof_type_osm,height_m,avg_tilt_deg,area_north_m2,area_east_m2,area_south_m2,area_west_m2,area_total_m2
0,DENW43AL00002dFC,unknown,NaN,4.27,0.00,115.16,102.35,0.0,217.51
1,DENW43AL00000k3W,unknown,NaN,45.23,74.39,0.00,52.67,0.0,127.06
2,DENW43AL00001j7h,mansard,7.728,30.08,36.07,0.00,55.15,0.0,91.22


---
## Step 5 — Join by Building ID

The M1 parquet has 'DEA_' added at the start of every building ID.
The roof dataset has the same ID without that prefix.

Example:
- Parquet: `DEA_DENW01AL1000CrNT`
- Roof   : `DENW01AL1000CrNT`

Stripping 'DEA_' allows a direct match — 99.3% of buildings matched.

In [7]:
# Strip DEA_ prefix from parquet IDs
m1['building_id'] = m1['id'].str.replace('DEA_', '', regex=False)

# Inner join — only buildings present in both datasets
merged = m1.merge(roofs, on='building_id', how='inner')

print(f'M1 buildings       : {len(m1):,}')
print(f'Roof buildings     : {len(roofs):,}')
print(f'Matched buildings  : {len(merged):,}  ({len(merged)/len(m1)*100:.1f}%)')
print(f'Unmatched in M1    : {len(m1)-len(merged):,}  (no roof data — dropped)')

M1 buildings       : 4,133,323
Roof buildings     : 5,097,348
Matched buildings  : 4,103,455  (99.3%)
Unmatched in M1    : 29,868  (no roof data — dropped)


---
## Step 6 — Roof Classification

Three categories based on average tilt angle:
- **Flat** (`avg_tilt_deg < 10`) — use `area_total_m2` for PV clustering
- **Gabled** (`avg_tilt_deg >= 10`) — cluster south and east-west separately
- **Unknown** (`NaN tilt`) — treated as gabled (most have orientation area values)

East and West areas are combined into one east-west variable because:
- A symmetric saddle roof has roughly equal east and west areas
- Combining reduces clustering variables without losing physical meaning

Buildings with only north facing area get `has_pv_area = False` — excluded from clustering.

In [ ]:
# East-West combined (north excluded)
merged['area_eastwest_m2'] = merged['area_east_m2'] + merged['area_west_m2']

# Classify roof type
merged['roof_category'] = 'gabled'
merged.loc[merged['avg_tilt_deg'] < FLAT_TILT_THRESHOLD, 'roof_category'] = 'flat'
merged.loc[merged['avg_tilt_deg'].isna(), 'roof_category'] = 'unknown'

# Flag buildings with no usable PV area (only north facing)
merged['has_pv_area'] = (
    (merged['area_south_m2'] > 0) | (merged['area_eastwest_m2'] > 0)
)

print('Roof categories:')
for cat, count in merged['roof_category'].value_counts().items():
    print(f'  {cat:<10}: {count:>10,}  ({count/len(merged)*100:.1f}%)')

print(f'\nBuildings with no usable PV area: {(~merged["has_pv_area"]).sum():,} ({(~merged["has_pv_area"]).mean()*100:.1f}%)')
print('These have only north-facing roof — excluded from PV clustering')

In [ ]:
# Roof category per building type
print('Roof category per building type:')
print(pd.crosstab(merged['size_class'], merged['roof_category']).to_string())
print('\nNote: AB has highest flat roof proportion (31%) — expected for large apartment blocks')

---
## Step 7 — Distribution Overview

Visualize south and east-west roof area distributions before clustering.
Shows why clustering is needed — wide spread of values across building types.

In [ ]:
BASE_DIR   = os.getcwd()  # folder where this notebook is located
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, sc in zip(axes.flat, SIZE_CLASSES):
    subset = merged[merged['size_class'] == sc]
    gabled = subset[subset['roof_category'] != 'flat']

    ax.hist(gabled['area_south_m2'].clip(upper=gabled['area_south_m2'].quantile(0.99)),
            bins=50, alpha=0.7, color='goldenrod', label='South area')
    ax.hist(gabled['area_eastwest_m2'].clip(upper=gabled['area_eastwest_m2'].quantile(0.99)),
            bins=50, alpha=0.7, color='steelblue', label='East-West area')
    ax.set_title(f'{sc} — gabled roofs only', fontsize=11, fontweight='bold')
    ax.set_xlabel('Roof area (m2)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Roof area distributions by building type (clipped at P99)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roof_distributions.pdf'), bbox_inches='tight')
plt.show()

---
## Step 8 — Helper Functions

### 8.1 Elbow Detection
Finds k where inertia improvement slows down most.
Draws line from k=2 to k=10 on inertia curve.
Selects k with maximum perpendicular distance from that line.

### 8.2 Cluster Variable
Full clustering pipeline for one roof area variable in one building type subset.
Produces elbow plot, boxplot, cluster labels, and statistics.

In [ ]:
def find_elbow(inertias, k_range):
    """
    Elbow detection via maximum perpendicular distance from reference line.
    Draws straight line from (k=2, inertia) to (k=10, inertia).
    Returns k with maximum distance — the bend point.
    """
    x1, y1 = k_range[0],  inertias[0]
    x2, y2 = k_range[-1], inertias[-1]
    dx, dy   = x2 - x1, y2 - y1
    line_len = np.sqrt(dx**2 + dy**2)
    distances = [
        abs(dy*k - dx*inertia + x2*y1 - y2*x1) / line_len
        for k, inertia in zip(k_range, inertias)
    ]
    return k_range[np.argmax(distances)]

print('find_elbow() ready')

In [ ]:
def cluster_variable(values, label, unit, size_class, output_dir):
    """
    Full clustering pipeline for one roof area variable.

    Steps:
    1. Clip at P99.9 — removes extreme outlier areas before scaling
    2. StandardScaler — normalises so m2 units do not skew distances
    3. Sample 30,000 for fast k-selection
    4. Test k=2 to 10 — record inertia and silhouette
    5. Elbow selects k
    6. Fit final model on ALL buildings
    7. Relabel: cluster 0 = smallest area, last = largest
    """
    n = len(values)
    if n < K_MIN * 10:
        print(f'  Skipping {label} for {size_class} — too few buildings ({n})')
        return np.zeros(n, dtype=int), pd.DataFrame(), 1

    cap   = np.percentile(values, 99.9)
    vals  = np.clip(values, 0, cap)
    X_all = StandardScaler().fit_transform(vals.reshape(-1, 1))

    np.random.seed(RANDOM_STATE)
    idx      = np.random.choice(n, min(SAMPLE_N, n), replace=False)
    X_sample = X_all[idx]

    k_range = list(range(K_MIN, min(K_MAX, n // 10) + 1))
    if len(k_range) < 2:
        k_range = [2]

    inertias, silhouettes = [], []
    for k in k_range:
        km  = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                              n_init=10, batch_size=5_000)
        lbl = km.fit_predict(X_sample)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X_sample, lbl,
                           sample_size=min(5_000, len(idx))))

    final_k    = find_elbow(inertias, k_range)
    best_sil_k = k_range[np.argmax(silhouettes)]
    print(f'  {label:<35} elbow k={final_k}  sil k={best_sil_k}  -> k={final_k}')

    # Elbow + silhouette plot
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax2 = ax1.twinx()
    ax1.plot(k_range, inertias,    'b-o', lw=2, ms=5, label='Inertia')
    ax2.plot(k_range, silhouettes, 'o-',  lw=2, ms=5, color='orange', label='Silhouette')
    ax1.axvline(x=final_k, color='red', ls='--', lw=2, alpha=0.7, label=f'Elbow k={final_k}')
    ax1.set_xlabel('Number of clusters (k)', fontsize=11)
    ax1.set_ylabel('Inertia — lower = tighter', fontsize=10, color='blue')
    ax2.set_ylabel('Silhouette — higher = better', fontsize=10, color='orange')
    ax1.set_title(f'Roof {size_class} — {label}  |  k={final_k}', fontsize=12, fontweight='bold')
    lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
    labs  = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
    ax1.legend(lines, labs, loc='upper right', fontsize=8)
    ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    safe = label.replace(' ', '_').replace('/', '_')
    plt.savefig(os.path.join(output_dir, f'elbow_{size_class}_{safe}.pdf'), bbox_inches='tight')
    plt.show()
    plt.close()

    # Fit on ALL buildings
    km_final       = MiniBatchKMeans(n_clusters=final_k, random_state=RANDOM_STATE,
                                     n_init=10, batch_size=5_000)
    cluster_labels = km_final.fit_predict(X_all)

    # Relabel: cluster 0 = smallest area, last = largest
    order = sorted(range(final_k), key=lambda c: values[cluster_labels == c].mean())
    remap = {old: new for new, old in enumerate(order)}
    cluster_labels = np.array([remap[c] for c in cluster_labels])

    # Stats per cluster
    stats = []
    for c in range(final_k):
        v = values[cluster_labels == c]
        stats.append({
            'size_class': size_class, 'variable': label, 'cluster': c,
            'n':      int(len(v)),
            'min':    round(float(v.min()), 1),
            'mean':   round(float(v.mean()), 1),
            'median': round(float(np.median(v)), 1),
            'max':    round(float(v.max()), 1),
            'std':    round(float(v.std()), 1),
        })

    # Boxplot — Box=middle 50% | Line=median | Whiskers=typical range | Circles=outliers
    fig, ax = plt.subplots(figsize=(max(6, final_k * 2), 5))
    colors  = plt.cm.tab10(np.linspace(0, 0.9, final_k))
    bp = ax.boxplot([values[cluster_labels == c] for c in range(final_k)],
                    patch_artist=True, medianprops={'color': 'black', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color); patch.set_alpha(0.8)
    for c, s in enumerate(stats):
        ax.text(c + 1, ax.get_ylim()[1],
                f"n={s['n']:,}\nmean={s['mean']:,.1f}",
                ha='center', va='top', fontsize=7,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    ax.set_xlabel('Cluster  (0 = smallest area, last = largest)', fontsize=11)
    ax.set_ylabel(f'{label} [{unit}]', fontsize=11)
    ax.set_title(f'Roof {size_class} — {label}  |  k={final_k}  |  n={n:,}',
                 fontsize=12, fontweight='bold')
    ax.set_xticklabels([f'Cluster {c}' for c in range(final_k)])
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'boxplot_{size_class}_{safe}.pdf'), bbox_inches='tight')
    plt.show()
    plt.close()

    return cluster_labels, pd.DataFrame(stats), final_k

print('cluster_variable() ready')

---
## Step 9 — Cluster per Building Type

For each building type separately:

**Gabled buildings with usable PV area:**
- Cluster `area_south_m2` independently
- Cluster `area_eastwest_m2` independently

**Flat roofs with usable PV area:**
- Cluster `area_total_m2` — orientation not applicable for flat roofs
- Flat clusters offset by +100 in the output to distinguish from gabled

**Buildings with no PV area (only north facing):**
- Assigned cluster = -1 — kept in dataset but excluded from PV clustering

In [ ]:
BASE_DIR   = os.getcwd()  # folder where this notebook is located
VARIABLES = [
    ('area_south_m2',    'South Roof Area',    'm2'),
    ('area_eastwest_m2', 'East-West Roof Area', 'm2'),
]

all_stats       = []
k_results       = {}
combo_per_type  = {}
all_assignments = []

for sc in SIZE_CLASSES:
    sc_all    = merged[merged['size_class'] == sc].copy().reset_index(drop=True)
    sc_gabled = sc_all[sc_all['roof_category'] != 'flat'].copy().reset_index(drop=True)
    sc_flat   = sc_all[sc_all['roof_category'] == 'flat'].copy().reset_index(drop=True)

    print(f'\n{"="*60}')
    print(f'  {sc}  ({len(sc_all):,} buildings total)')
    print(f'  Gabled: {len(sc_gabled):,}  |  Flat: {len(sc_flat):,}  |  No PV area: {(~sc_all["has_pv_area"]).sum():,}')
    print(f'{"="*60}')

    k_results[sc] = {}
    type_dir = os.path.join(OUTPUT_DIR, sc)
    os.makedirs(type_dir, exist_ok=True)

    # Initialise cluster columns — -1 = no usable area
    sc_all['cluster_south']    = -1
    sc_all['cluster_eastwest'] = -1

    # ── Gabled buildings with usable PV area ──────────────────────────
    gabled_pv = sc_gabled[sc_gabled['has_pv_area']].copy().reset_index(drop=True)
    print(f'  Clustering gabled buildings with usable PV area: {len(gabled_pv):,}')

    gabled_k = {}
    for col, label, unit in VARIABLES:
        vals = gabled_pv[col].values.astype(float)
        lbl, stats, k = cluster_variable(vals, label, unit, f'{sc}_gabled', type_dir)
        gabled_pv[f'cluster_{col}'] = lbl
        all_stats.append(stats)
        gabled_k[col] = k

    # Map back to sc_all
    gabled_mask = (sc_all['roof_category'] != 'flat') & sc_all['has_pv_area']
    sc_all.loc[gabled_mask, 'cluster_south']    = gabled_pv['cluster_area_south_m2'].values
    sc_all.loc[gabled_mask, 'cluster_eastwest'] = gabled_pv['cluster_area_eastwest_m2'].values

    k_results[sc]['gabled_south']    = gabled_k.get('area_south_m2', 1)
    k_results[sc]['gabled_eastwest'] = gabled_k.get('area_eastwest_m2', 1)

    # ── Flat roofs ────────────────────────────────────────────────────
    flat_pv = sc_flat[sc_flat['has_pv_area']].copy().reset_index(drop=True)
    if len(flat_pv) >= K_MIN * 10:
        print(f'  Clustering flat roofs: {len(flat_pv):,}')
        vals_flat = flat_pv['area_total_m2'].values.astype(float)
        lbl_flat, stats_flat, k_flat = cluster_variable(
            vals_flat, 'Flat Roof Total Area', 'm2', f'{sc}_flat', type_dir
        )
        flat_pv['cluster_flat'] = lbl_flat
        all_stats.append(stats_flat)
        k_results[sc]['flat_total'] = k_flat

        flat_mask = (sc_all['roof_category'] == 'flat') & sc_all['has_pv_area']
        sc_all.loc[flat_mask, 'cluster_south'] = lbl_flat + 100  # offset to distinguish
    else:
        print(f'  Flat roofs too few to cluster ({len(flat_pv)}) — assigned -2')
        k_results[sc]['flat_total'] = 0

    # ── Combinations ─────────────────────────────────────────────────
    k_s   = k_results[sc].get('gabled_south', 1)
    k_e   = k_results[sc].get('gabled_eastwest', 1)
    k_f   = k_results[sc].get('flat_total', 1) if k_results[sc].get('flat_total', 0) > 0 else 1
    combo = k_s * k_e + k_f
    combo_per_type[sc] = combo
    print(f'\n  Gabled combinations : {k_s} x {k_e} = {k_s*k_e}')
    print(f'  Flat combinations   : {k_f}')
    print(f'  Total for {sc}       : {combo}')

    # Save assignments
    export_cols = ['building_id', 'size_class', 'roof_category', 'has_pv_area',
                   'avg_tilt_deg', 'area_south_m2', 'area_eastwest_m2',
                   'area_total_m2', 'cluster_south', 'cluster_eastwest']
    sc_all[[c for c in export_cols if c in sc_all.columns]].to_csv(
        os.path.join(type_dir, f'Roof_{sc}_assignments.csv'), index=False
    )
    all_assignments.append(sc_all)
    print(f'  Saved: Roof_{sc}_assignments.csv')

---
## Step 10 — Combination Summary

Roof combinations are structured differently from energy demand combinations:
- **Gabled**: south clusters × east-west clusters (two independent dimensions)
- **Flat**: separate clusters using total area (no orientation dimension)

These multiply with the energy demand combinations to give total archetype count.

In [ ]:
total_roof = sum(combo_per_type.values())

print('COMBINATION SUMMARY')
print('='*60)
print(f'\n  Roof area combinations per building type:')
for sc in SIZE_CLASSES:
    k_s = k_results[sc].get('gabled_south', 1)
    k_e = k_results[sc].get('gabled_eastwest', 1)
    k_f = k_results[sc].get('flat_total', 0)
    print(f'  {sc}: gabled {k_s}x{k_e}={k_s*k_e}  +  flat {k_f}  =  {combo_per_type[sc]}')

print(f'\n  Total roof combinations: {total_roof}')
print(f'\n  Combined with energy demand clustering:')
print(f'    M1  (292 combinations) x {total_roof} = {292*total_roof:,} total archetypes')
print(f'    M3b (267 combinations) x {total_roof} = {267*total_roof:,} total archetypes')
print(f'  M1 combined: {"WITHIN" if 292*total_roof <= 65000 else "ABOVE"} target')

---
## Step 11 — Gap Analysis

Checks whether adjacent clusters are meaningfully different.
Gap = percentage difference between adjacent cluster means.
Threshold: 20% minimum.

**Note on TH south area C0 -> C1 (0% gap):**
Cluster 0 contains TH buildings with zero south area — they face only east-west.
The gap is 0% but this is physically meaningful not a clustering error.
Cluster 0 = no south roof at all, Cluster 1 = small south roof.
These are genuinely different building situations for SESMG.

In [ ]:
stats_df = pd.concat(all_stats, ignore_index=True)

print('GAP ANALYSIS — relative difference between adjacent cluster means')
print('='*60)

all_gaps_ok = True
for sc in SIZE_CLASSES:
    print(f'\n  {sc}:')
    for col, label, _ in VARIABLES:
        sc_label = f'{sc}_gabled'
        sc_stats = stats_df[
            (stats_df['size_class'] == sc_label) &
            (stats_df['variable'] == label)
        ].sort_values('cluster')
        if sc_stats.empty:
            continue
        for i in range(len(sc_stats) - 1):
            m0  = sc_stats.iloc[i]['mean']
            m1v = sc_stats.iloc[i + 1]['mean']
            gap = (m1v - m0) / m0 * 100 if m0 > 0 else 0
            flag = '' if gap >= 20 else '  <- SMALL GAP (see note above)'
            if gap < 20:
                all_gaps_ok = False
            print(f'  {label[:22]:<24} C{i}->C{i+1}: {m0:>8,.1f} -> {m1v:>8,.1f} m2  ({gap:.0f}%){flag}')

if all_gaps_ok:
    print('\n  All gaps >= 20% — clusters are well separated')
else:
    print('\n  Note: gaps < 20% are physically meaningful — see TH south area explanation above')

---
## Step 12 — Save Excel Summary

In [ ]:
BASE_DIR   = os.getcwd()  # folder where this notebook is located
excel_path = os.path.join(OUTPUT_DIR, 'Roof_clustering_by_type_summary.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

    # Combination summary
    rows = []
    for sc in SIZE_CLASSES:
        rows.append({
            'Size class':     sc,
            'Gabled south k': k_results[sc].get('gabled_south', 1),
            'Gabled EW k':    k_results[sc].get('gabled_eastwest', 1),
            'Gabled combos':  k_results[sc].get('gabled_south', 1) * k_results[sc].get('gabled_eastwest', 1),
            'Flat k':         k_results[sc].get('flat_total', 0),
            'Total combos':   combo_per_type[sc],
        })
    rows.append({'Size class': 'TOTAL', 'Total combos': total_roof})
    pd.DataFrame(rows).to_excel(writer, sheet_name='Combinations', index=False)

    # All cluster stats
    stats_df.to_excel(writer, sheet_name='All Stats', index=False)

    # Per building type
    for sc in SIZE_CLASSES:
        sc_stats = stats_df[stats_df['size_class'].str.startswith(sc)]
        if not sc_stats.empty:
            sc_stats.to_excel(writer, sheet_name=f'Stats {sc}', index=False)

print(f'Saved: {excel_path}')
print(f'Output directory: {OUTPUT_DIR}')
print('\nDone.')